# APEX-seq (Fazal et al. 2019) vs unlabelled controls: fracSanity, tandem poly(A) site usage and APA

The analyses of `ONT_organelles.fracSanity.ipynb` (ONT, FACS-sorted subcellular fractions of HCT116) on the public APEX-seq data of Fazal et al. (2019, *Cell* 178:473, GEO GSE116008; HEK293T, Illumina TruSeq stranded poly(A)+, 2 × 75 nt). Read QC, nuclear RNA, compression and library complexity of these libraries are in `APEX_seq_vs_Control.ipynb`; the comparison with the ONT fractions is in `APEX_vs_ONT_organelles.fracSanity.ipynb`.

**Fractions and reference.** A fraction is the labelled, streptavidin-enriched RNA near an APEX2 construct (two biological replicates each): NLS (nucleus), NIK (nucleolus), LMA (nuclear lamina), NucPore (nuclear pore), NES (cytosol), ERM (ER membrane, cytosolic face), KDEL (ER lumen), OMM (outer mitochondrial membrane), MITO (mitochondrial matrix). The reference ("Total") is the 18 unlabelled controls (no H2O2, otherwise identical including the enrichment; all constructs together, as in the 18-control analysis of the paper). The nuclear and cytosolic fractions of the paper (biochemical fractionation, not enriched; two each) are included as two more fractions against the same reference.

**Quantities** (as in the ONT notebook).
- $\alpha_g$: fraction of the mRNA molecules of gene $g$ in the cell that are labelled and recovered with the construct, up to the global factor $F$ (fracSanity). For APEX-seq, $\alpha_g$ is a labelling probability: proximity to the construct, not a physical compartment.
- $F$: abundance-weighted mean of the $\alpha_g$; weakly determined without spike-ins. The ranking of genes within a fraction, $D_g = \log_2\alpha_g - \log_2 F$ = log2 TPM in the fraction minus log2 TPM in the controls, is well determined.
- Over- and under-representation of gene sets in each fraction: ISMARA-style activities on the per-library $\log_2\alpha_g$.
- Tandem poly(A) sites (PAS) of terminal exons (PAQR, from read coverage): relative usage, 3' UTR shortening / lengthening against the controls, and $\alpha$ of every PAS (fracSanity on the PAS counts).

**Design choices.**
- **Gene universe: all genes of GENCODE v42 basic** with at least one read pair (featureCounts, unique alignments, reverse-stranded).
- **Length correction: log2 TPM.** Short reads come from along the fragmented transcript, so the read count of a gene grows with the length of its mRNA. Sanity is run with the exonic length of every gene (`sanity_normalization(..., gene_lengths=...)`, union of the exons as counted by featureCounts), so its log2 expression is log2 TPM, the share of mRNA molecules. The ONT reads are unfragmented full-length molecules (one read per molecule), so this **log2 TPM corresponds to the log2 CPM of the ONT notebook**. The length factor would cancel in $D_g$ of a gene, but not in the ranking of genes by expression (reference expression, expression deciles of the gene-set analysis, the ≥ 10 TPM threshold of the calls).
- **Replicates are not paired**: each fraction (average of its two targets, or one target library) is compared with the average of the 18 controls. MITO T2 and its technical replicate are added up.
- Sanity with `v_method='EAP'`, ISMARA replicate averages with `method='reml'`, fracSanity with $\pi \le 0.5$ and a uniform prior on $F$, as for the ONT data.
- **PAS counts are PAQR mean coverages** of the tandem PAS of terminal exons (`tandem_pas_expression-mean`), rounded, as in `NFYA_project.ipynb`; terminal exons with a missing value (−1) in any library are left out. They are proportional to, but not equal to, read counts, so their Poisson noise is approximate.

The heavy steps run in `WF/scripts/APEX_fracSanity_pipeline.py` (SLURM job `APEX_fracSanity.sbatch`); this notebook reads their outputs and fits the gene-set activities.

**Kernel.** `sanity_utils` conda environment.

In [ ]:
import warnings
warnings.simplefilter('ignore')

import os
import re
import csv
import glob
import pickle

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import spearmanr

from dotenv import load_dotenv

import seaborn as sns; sns.set()
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, HTML
%matplotlib inline

import sanity_utils as su
from sanity_utils import plotting as su_plotting
from sanity_utils.genesets import (GeneSetCollection, set_source_table, load_go, load_reactome, load_kegg,
                                   load_wikipathways, read_ncbi_gene_info, latest_release)
print("sanity_utils", su.__version__, "| Sanity", su.get_sanity_version(), su.get_sanity_commit())

In [ ]:
load_dotenv("APA_localization.scicore.env", override=True)
subdirs = {"wf_runs_dir": os.getenv("WF_RUNS_DIR"), "figures_dir": os.getenv("FIGURES_DIR"), "tables_dir": os.getenv("TABLES_DIR"),
           "external_data_dir": os.getenv("EXTERNAL_DATA_DIR")}

ANALYSIS = "APEX_seq_fracSanity"
SU_OUT = os.path.join(subdirs["wf_runs_dir"], "Sanity_utils_APEX_seq_Fazal2019")   # WF/scripts/APEX_fracSanity_pipeline.py
PAS_LEVEL = "pas"
GENESETS_DIR = os.path.join(subdirs["external_data_dir"], "genesets_human")           # sanity-utils-fetch-genesets all --organism human
if os.environ.get("APEX_FRACSANITY_TEST_DIR"):
    # test runs of the notebook on a reduced data set: inputs and outputs in a separate folder
    _test = os.environ["APEX_FRACSANITY_TEST_DIR"]
    SU_OUT, subdirs["figures_dir"], subdirs["tables_dir"] = (os.path.join(_test, d) for d in ("out", "figures", "tables"))
FIGS = os.path.join(subdirs["figures_dir"], "gene_expression_analysis", ANALYSIS)
TABLES = os.path.join(subdirs["tables_dir"], "gene_expression_analysis", ANALYSIS)

def figure_caption(path, formats=("png", "pdf")):
    # file name of a saved figure, shown with the plot; the links point to the full-size files relative to the
    # HTML report (WF/scripts/make_html_reports.py copies them to <report>/figures/, same subfolders as FIGURES_DIR)
    path = os.path.splitext(path)[0] if path.endswith((".png", ".pdf")) else path
    rel = os.path.relpath(path, subdirs["figures_dir"])
    links = " | ".join(f'<a href="figures/{rel}.{f}" target="_blank">{f}</a>' for f in formats)
    display(HTML(f'<p><b>Figure:</b> <code>{os.path.basename(path)}</code> &nbsp;({links})</p>'))

def savefig(fig, subdir, name, formats=("png", "pdf"), dpi=300):
    out = os.path.join(FIGS, subdir)
    os.makedirs(out, exist_ok=True)
    for fmt in formats:
        fig.savefig(os.path.join(out, f"{name}.{fmt}"), bbox_inches="tight", dpi=dpi)
    figure_caption(os.path.join(out, name), formats)

def save_tsv(df, subdir, name, index=False):
    out = os.path.join(TABLES, subdir)
    os.makedirs(out, exist_ok=True)
    df.to_csv(os.path.join(out, name), sep="\t", header=True, index=index, quoting=csv.QUOTE_NONE)
    return os.path.join(out, name)

def load_pickle(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)

def safe(name):
    # file name of a cohort or library (as in the pipeline)
    return re.sub(r"[^A-Za-z0-9]+", "_", name)

# notation as in APEX_seq_vs_Control.ipynb: colour = construct (fraction); marker = batch
FRACTION_ORDER = ["Total", "NLS", "NIK", "LMA", "NucPore", "NES", "ERM", "KDEL", "OMM", "MITO", "Nuc", "Cyto"]
FRACTION_COLORS = {"Total": "black", "NLS": "tab:red", "NIK": "darkred", "LMA": "tab:orange", "NucPore": "goldenrod", "NES": "tab:blue",
                   "ERM": "tab:green", "KDEL": "yellowgreen", "OMM": "tab:purple", "MITO": "tab:pink", "Nuc": "tab:brown", "Cyto": "tab:cyan"}
BATCH_ORDER = ["Controls", "APEX", "Fractionation"]
BATCH_MARKERS = {"Controls": "D", "APEX": "o", "Fractionation": "s"}
BATCH_COLORS = {"Controls": "black", "APEX": "0.6", "Fractionation": "0.3"}

def notation_handles(fractions, batches):
    return ([matplotlib.lines.Line2D([], [], color=FRACTION_COLORS[f], marker="o", ls="", label=f) for f in FRACTION_ORDER if f in set(fractions)]
            + [matplotlib.lines.Line2D([], [], color="grey", marker=BATCH_MARKERS[b], ls="", label=b) for b in BATCH_ORDER if b in set(batches)])

def color_ticklabels(labels, fractions):
    for t, f in zip(labels, fractions):
        t.set_color(FRACTION_COLORS.get(f, "black"))
os.makedirs(FIGS, exist_ok=True); os.makedirs(TABLES, exist_ok=True)
print(SU_OUT)

## Pipeline

`WF/scripts/APEX_fracSanity_pipeline.py`, run by `sbatch APEX_fracSanity.sbatch` (steps whose outputs exist are skipped, nothing is overwritten), on the outputs of the bulk RNA-seq workflow (`wf_runs/APEX_seq_Fazal2019`, `WF/scripts/run_bulk_workflow.sh`):
1. `prepare`: sample table, gene counts (featureCounts, `gencode.v42.basic`, read pairs), tandem PAS of terminal exons (PAQR), PAS annotation (3' UTR length);
2. `complexity`: distinct fragments per library (for `APEX_seq_vs_Control.ipynb`);
3. `sanity`: C++ Sanity (EAP) on genes and PAS, separately for the controls and the fractions, and jointly on the PAS (relative usage and APA); ISMARA replicate averages per cohort (`reml`);
4. `fit_cohorts`: fracSanity per cohort, cohort average vs control average, genes and PAS;
5. `fit_replicates`: fracSanity of every fraction library vs the control average (genes), the input of the gene-set analysis;
6. `usage`: relative usage of each PAS against the other PAS of its terminal exon, usage-weighted mean 3' UTR length, most proximal vs distal PAS; per cohort vs controls.

In [ ]:
print(f"sbatch APEX_fracSanity.sbatch   # outputs in {SU_OUT}")
print(f"log: {sorted(glob.glob('APEX_fracSanity_*.log'))[-1:] }")

## Samples

The 18 unlabelled controls are the reference (cohort `Total`); every APEX construct is a cohort (2 targets), and the nuclear and cytosolic fractionation libraries are two more cohorts.

In [ ]:
meta_all = pd.read_csv(os.path.join(SU_OUT, "inputs", "metadata.tsv"), sep="\t")
save_tsv(meta_all, "samples", "samples.tsv")
display(meta_all[["sample", "batch", "fraction", "cohort", "replicate", "library", "assigned_read_pairs", "use"]])
meta = meta_all.loc[meta_all["use"]].reset_index(drop=True)

def cohort_sort_key(c):
    r = meta.loc[meta["cohort"] == c].iloc[0]
    return (BATCH_ORDER.index(r["batch"]), FRACTION_ORDER.index(r["fraction"]))
COHORTS = sorted(meta.loc[meta["fraction"] != "Total", "cohort"].unique(), key=cohort_sort_key)
cohort_info = meta.groupby("cohort")[["batch", "fraction"]].first().assign(n_libraries=meta.groupby("cohort").size())
cohort_info = cohort_info.loc[["Total"] + COHORTS]
display(cohort_info)

genes_df = pd.read_csv(os.path.join(SU_OUT, "inputs", "genes.tsv"), sep="\t", index_col=0)
gene_names = genes_df["gene_name"].fillna(pd.Series(genes_df.index, index=genes_df.index))
# biotype groups: mitochondrially encoded genes apart
biotype = genes_df["gene_type"].where(genes_df["chrom"] != "chrM", "mitochondrially_encoded")

## Sanity normalization

C++ Sanity via Sanity_utils, `v_method='EAP'`, separately on the 18 controls and on all fraction libraries, with length correction (log2 TPM over all genes). Below: inferred biological variance $v_g$ against expression, and a PCA of the log2 TPM of genes with reads in all libraries (Total and fractions; the two Sanity runs are normalised separately, both to TPM).

In [ ]:
LEVEL_PARTS = {"genes": ("Total", "fractions"), PAS_LEVEL: ("Total", "fractions", "joint")}
sanity = {(lvl, part): load_pickle(os.path.join(SU_OUT, "sanity", lvl, part, "sanity.pkl")) for lvl, parts in LEVEL_PARTS.items() for part in parts}
for (lvl, part), d in sanity.items():
    r = d["result"]
    print(f"{lvl:5s} {part:9s}: {r.log2_expression.shape[0]:7,} features x {r.log2_expression.shape[1]:2d} libraries | {r.params.get('units')} | "
          f"v_method {r.params.get('v_method')}")

sns.set(font_scale=0.9, style="ticks")
for part in ("Total", "fractions"):
    fig = su_plotting.plot_variance_vs_expression(sanity[("genes", part)]["result"])
    savefig(fig, "QC", f"variance_vs_expression.genes.{part}")
    plt.show()

# PCA: log2 TPM of genes with reads in all used libraries, Total and fractions together
gene_counts = pd.read_csv(os.path.join(SU_OUT, "inputs", "gene_counts.tsv.gz"), sep="\t", index_col=0)
expressed = gene_counts.index[(gene_counts[meta["sample"]] > 0).all(axis=1)]
X = pd.concat([sanity[("genes", p)]["result"].log2_expression for p in ("Total", "fractions")], axis=1).reindex(expressed)[meta["sample"]].dropna()
Xc = X.sub(X.mean(axis=1), axis=0).to_numpy()
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
var_expl = S ** 2 / np.sum(S ** 2)
pcs = pd.DataFrame(Vt[:3].T * S[:3], index=X.columns, columns=["PC1", "PC2", "PC3"]).join(meta.set_index("sample")[["batch", "fraction", "replicate"]])
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (a, b) in zip(axes, (("PC1", "PC2"), ("PC1", "PC3"))):
    for s, r in pcs.iterrows():
        ax.scatter(r[a], r[b], color=FRACTION_COLORS[r["fraction"]], marker=BATCH_MARKERS[r["batch"]], s=50, edgecolor="black", lw=0.4)
        ax.annotate(r["replicate"] if r["fraction"] == "Total" else f'{r["fraction"]} {r["replicate"]}', (r[a], r[b]), fontsize=5.5,
                    xytext=(3, 3), textcoords="offset points")
    ax.set(xlabel=f"{a} ({var_expl[int(a[-1]) - 1]:.0%})", ylabel=f"{b} ({var_expl[int(b[-1]) - 1]:.0%})")
axes[1].legend(handles=notation_handles(pcs["fraction"], pcs["batch"]), fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1))
axes[0].set_title(f"{len(X):,} genes with reads in all {X.shape[1]} libraries", fontsize=9)
sns.despine(); fig.tight_layout()
savefig(fig, "QC", "PCA.genes.log2TPM")
plt.show()

# normalized expression tables: log2 TPM per library and cohort averages, with gene annotation
annot = genes_df[["gene_name", "gene_type", "chrom"]]
log2cpm = pd.concat([sanity[("genes", p)]["result"].log2_expression for p in ("Total", "fractions")], axis=1)
save_tsv(annot.join(log2cpm, how="right").rename_axis("gene_id").reset_index(), "normalized_expression", "Sanity_log2TPM.genes.per_library.tsv.gz")
avg_g = {p: sanity[("genes", p)]["averages"] for p in ("Total", "fractions")}
avg_tab = pd.concat([avg_g[p].means.add_prefix("mean;").join(avg_g[p].errors.add_prefix("error;")) for p in ("Total", "fractions")], axis=1)
save_tsv(annot.join(avg_tab, how="right").rename_axis("gene_id").reset_index(), "normalized_expression", "Sanity_log2TPM.genes.cohort_averages.tsv.gz")

### Interpretation of the Sanity normalization

- **Features.** 31,593 genes have reads in the 18 controls and 39,711 in the 22 fraction libraries; 29,966 have reads in both and enter fracSanity. At the PAS level, 5,735–5,792 tandem PAS (Total, fractions, joint run).
- **PC1 (38%, 10,658 genes with reads in all 40 libraries) separates APEX MITO and the cytosolic fractionation libraries from the rest.** The MITO libraries are dominated by mitochondrially encoded RNA (37–62% of the assigned reads, `APEX_seq_vs_Control.ipynb`), and Cyto lacks the nuclear RNA of the controls. The controls form a tight group (NES C1 apart).
- **The replicates of NIK, ERM and MITO disagree.** NIK T1 and ERM T1 lie far from their T2 replicates on PC2 and PC3, and so does MITO T2 from T1. NIK T1 and ERM T1 are the two libraries with the lowest complexity (1.7% and 2.0% distinct fragments) and the steepest slopes against the controls (2.7 and 2.3, `APEX_seq_vs_Control.ipynb`): PCR jackpots rather than localization. NLS, LMA, NES, KDEL, OMM and the fractionation replicates agree.

## fracSanity: global fraction $F$ and the shape of the fraction vs Total

For each cohort, $D_g = \langle A^{\text{fraction}}_g\rangle - \langle A^{\text{Total}}_g\rangle = \log_2\alpha_g - \log_2 F + \varepsilon_g$, with $\alpha_g \sim$ Beta$(a, b)$, the contaminated-Gaussian error model with $\pi \le 0.5$, and a uniform prior on $F$; profile likelihood over $\log_2 F$, $F$ reported as posterior median with 95% interval. Genes: all genes with reads in both the Total and the fraction Sanity run.

If every gene had the same $\alpha_g$, the log2 TPM distribution of the fraction would equal that of the controls up to noise; the densities below show how the shapes differ (compare the compression analysis of `APEX_seq_vs_Control.ipynb`, on nuclear protein-coding genes with ≥ 10 TPM in the controls).

In [ ]:
fits = {lvl: {c: load_pickle(os.path.join(SU_OUT, "fracSanity", lvl, "cohorts", safe(c) + ".pkl")) for c in COHORTS} for lvl in ("genes", PAS_LEVEL)}
F_df = pd.DataFrame([{"level": lvl, "cohort": c, "batch": cohort_info.loc[c, "batch"], "fraction": cohort_info.loc[c, "fraction"],
                      "n_libraries": cohort_info.loc[c, "n_libraries"], **{k: f[k] for k in ("F", "F_CI_lower", "F_CI_upper", "F_MLE", "log2F_mean",
                      "var_log2_F", "F_identifiable", "a", "b", "pi_outlier", "tau", "ks_stat")}, "n_features": len(f["gene_ids"])}
                     for lvl in fits for c, f in fits[lvl].items()])
save_tsv(F_df, "fracSanity", "fracSanity_global_F.tsv")
display(F_df.round(4))

sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(9, 3.8))
for k, lvl in enumerate(("genes", PAS_LEVEL)):
    d = F_df.loc[F_df["level"] == lvl].set_index("cohort").loc[COHORTS]
    for i, (c, r) in enumerate(d.iterrows()):
        ax.errorbar(i + (k - 0.5) * 0.3, r["F"], yerr=[[r["F"] - r["F_CI_lower"]], [r["F_CI_upper"] - r["F"]]], fmt=BATCH_MARKERS[r["batch"]],
                    color=FRACTION_COLORS[r["fraction"]], mfc=FRACTION_COLORS[r["fraction"]] if lvl == "genes" else "white", mec=FRACTION_COLORS[r["fraction"]],
                    ecolor="grey", ms=7, capsize=2)
ax.set_yscale("log")
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=60, ha="right", fontsize=8)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(ylabel="$F$ (posterior median, 95% interval)")
ax.legend(handles=[matplotlib.lines.Line2D([], [], color="grey", marker="o", ls="", label="genes"),
                   matplotlib.lines.Line2D([], [], color="grey", marker="o", mfc="white", ls="", label="tandem PAS")],
          fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
sns.despine(); fig.tight_layout()
savefig(fig, "fracSanity", "globalF_per_cohort")
plt.show()

# densities of log2 TPM in Total and in each cohort (genes in both)
tot_av, frac_av = sanity[("genes", "Total")]["averages"], sanity[("genes", "fractions")]["averages"]
common = tot_av.means.index.intersection(frac_av.means.index)
t_all = tot_av.means.loc[common, "Total"].to_numpy()
grid = np.linspace(-6, 18, 300)
ncol = 6
nrow = int(np.ceil(len(COHORTS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.2 * nrow), sharex=True, sharey=True, squeeze=False)
shape_rows = []
for ax, c in zip(axes.flat, COHORTS):
    f = cohort_info.loc[c, "fraction"]
    p = frac_av.means.loc[common, c].to_numpy()
    ax.plot(grid, stats.gaussian_kde(t_all)(grid), color="black", lw=1.1)
    ax.plot(grid, stats.gaussian_kde(p)(grid), color=FRACTION_COLORS[f], lw=1.6)
    ax.set_title(c, fontsize=8)
    shape_rows.append({"cohort": c, "sd_Total": t_all.std(), "sd_fraction": p.std(), "median_Total": np.median(t_all), "median_fraction": np.median(p),
                       "KS_fraction_vs_Total": stats.ks_2samp(p, t_all).statistic, "var_D": np.var(p - t_all), "spearman_D_vs_Total": spearmanr(p - t_all, t_all)[0]})
for ax in axes.flat[len(COHORTS):]:
    ax.axis("off")
for ax in axes[-1]:
    ax.set_xlabel("log2 TPM")
for ax in axes[:, 0]:
    ax.set_ylabel("density")
sns.despine(); fig.tight_layout()
savefig(fig, "fracSanity", "fraction_vs_Total.density.all_genes")
plt.show()
shape_df = pd.DataFrame(shape_rows)
save_tsv(shape_df, "fracSanity", "fraction_vs_Total.shape_stats.all_genes.tsv")
display(shape_df.round(3))

# fit diagnostics per cohort (profile likelihood, PIT histogram, alpha_g vs the fitted Beta), saved only
for lvl in ("genes", PAS_LEVEL):
    for c, f in fits[lvl].items():
        fig = su_plotting.plot_frac_sanity_fit(f, condition_name=f"{c} ({lvl})")
        out = os.path.join(FIGS, "fracSanity", f"fit_diagnostics_{lvl}")
        os.makedirs(out, exist_ok=True)
        fig.savefig(os.path.join(out, f"fit.{safe(c)}.png"), bbox_inches="tight", dpi=150)
        plt.close(fig)
print("fit diagnostics:", os.path.join(FIGS, "fracSanity", "fit_diagnostics_genes"), f"and ..._{PAS_LEVEL}")

### Interpretation of $F$ and of the shapes

- **$F$ is not identified in any of the 22 fits** (11 cohorts × genes / PAS). The MLE is at the lower bound in all of them, the posterior medians (0.0005–0.11) follow the prior, and the 95% intervals span 1.5–3 orders of magnitude. For APEX-seq this is expected: $F$ would be the share of the cell's RNA that is biotinylated and recovered, which the counts of a library cannot tell.
- **Why it is not identified.** The fitted Beta distributions put all $\alpha_g$ far below 1 (a = 0.9–50, b = 20–4,700; mean $a/(a+b)$ ≤ 0.07, except the PAS fit of NucPore, 0.23). The bound $\alpha_g \le 1$, the only feature of the model that fixes $F$, is then never reached. Equivalently, the distribution of $D_g$ has no sharp upper edge at which genes pile up. At $F \to 0$ the Beta becomes a scaled Gamma, whose log has a light left tail (rate $a$) and an even lighter right tail. This is the source of the shrinkage described in the next section.
- **The outlier fraction is at its bound $\pi$ = 0.5 for the nuclear and cytosolic fractionation at the gene level** (and in 6 of 11 PAS fits), and 0.29–0.39 for NLS, LMA and NES. The error model then explains the spread of $D_g$ by noise rather than by $\alpha_g$.
- **Shapes.** $D_g$ is most spread in the nuclear fraction (var 6.95) and in NLS and LMA (5.9), least in NucPore and OMM (2.8–2.9). The KS distance of the log2 TPM distribution to the controls is largest for Nuc (0.22), NLS, NES and LMA (0.16), and smallest for NucPore (0.05) and NIK (0.06). $D_g$ decreases with expression in the controls in every cohort (Spearman −0.22 to −0.64), most in the nuclear fraction and in MITO. Over all genes this is mostly the many weakly expressed genes, whose log2 TPM Sanity shrinks.

## $\alpha_g$ of individual genes: over- and under-represented mRNAs per fraction

$\alpha_g$ per cohort (posterior median, 95% interval, uncertainty of $F$ included). Because $F$ is weakly determined, genes are compared **within a cohort relative to its average gene**: $r_g = \log_2\alpha_g - \text{median}_g \log_2\alpha_g$ (posterior mean of $\log_2\alpha_g$), the same for any value of $F$. The 95% intervals of $\alpha_g$ are dominated by the uncertainty of $F$, which all genes of a cohort share (posterior s.d. of $\log_2\alpha_g$ ≈ 1.9); as in the P-body analysis, the calls therefore use the gene-specific variance, the posterior variance of $\log_2\alpha_g$ minus that of $\log_2 F$, at least the measurement variance of $D_g$, and a normal approximation: local false sign rate $\text{lfsr}_g = \Phi(-|r_g| / s_g)$. A gene is called **over-represented** in a fraction if $r_g > 0.5$ and lfsr < 0.05, **under-represented** if $r_g < -0.5$ and lfsr < 0.05; only genes with ≥ 10 TPM in Total are called. The posterior means are shrunk towards the Beta distribution of the $\alpha_g$ (slope of $r_g$ on the centred $D_g$ 0.2–0.7), so $r_g$ is smaller than the raw log2 ratio of fraction and Total.

In [ ]:
from scipy.stats import norm
MIN_TPM_CALL, R_CUT, LFSR_CUT = 10, 0.5, 0.05
total_cpm = 2 ** tot_av.means["Total"]
var_log2F = F_df.loc[F_df["level"] == "genes"].set_index("cohort")["var_log2_F"]
alpha_rows = []
for c, f in fits["genes"].items():
    a = pd.DataFrame({"gene_id": f["gene_ids"], "alpha": f["alpha_g"], "alpha_CI_lower": f["alpha_g_CI_lower"], "alpha_CI_upper": f["alpha_g_CI_upper"],
                      "log2_alpha_mean": f["log2_alpha_mean"], "log2_alpha_sd": np.sqrt(f["log2_alpha_var"])})
    a["D"] = (frac_av.means[c] - tot_av.means["Total"]).reindex(a["gene_id"]).to_numpy()
    a["r_rel"] = a["log2_alpha_mean"] - np.median(a["log2_alpha_mean"])
    a["Total_TPM"] = total_cpm.reindex(a["gene_id"]).to_numpy()
    # gene-specific s.d.: posterior variance minus the variance of log2 F (shared by all genes), at least var(D_g)
    var_D = (frac_av.errors[c] ** 2 + tot_av.errors["Total"] ** 2).reindex(a["gene_id"]).to_numpy()
    a["sd_rel"] = np.sqrt(np.maximum(a["log2_alpha_sd"] ** 2 - var_log2F[c], var_D))
    a["lfsr_rel"] = norm.cdf(-a["r_rel"].abs() / a["sd_rel"])
    sig = (a["Total_TPM"] >= MIN_TPM_CALL) & (a["lfsr_rel"] < LFSR_CUT)
    a["call"] = np.select([sig & (a["r_rel"] > R_CUT), sig & (a["r_rel"] < -R_CUT)], ["over", "under"], "n.s.")
    alpha_rows.append(a.assign(cohort=c, batch=cohort_info.loc[c, "batch"], fraction=cohort_info.loc[c, "fraction"]))
alpha_df = pd.concat(alpha_rows, ignore_index=True)
alpha_df.insert(1, "gene_name", gene_names.reindex(alpha_df["gene_id"]).to_numpy())
alpha_df.insert(2, "gene_type", biotype.reindex(alpha_df["gene_id"]).to_numpy())
save_tsv(alpha_df, "alpha_genes", "fracSanity_alpha.genes.per_cohort.tsv.gz")
n_calls = alpha_df.groupby("cohort")["call"].value_counts().unstack(fill_value=0).reindex(COHORTS)
display(n_calls)

# distribution of alpha_g per cohort with F
sns.set(font_scale=0.9, style="ticks")
fig, ax = plt.subplots(figsize=(10, 3.8))
d = alpha_df.assign(log10_alpha=np.log10(alpha_df["alpha"]))
sns.violinplot(data=d, x="cohort", y="log10_alpha", order=COHORTS, hue="fraction", palette=FRACTION_COLORS, cut=0, inner="quart", ax=ax,
               legend=False, density_norm="width")
Fg = F_df.loc[F_df["level"] == "genes"].set_index("cohort")["F"]
ax.scatter(range(len(COHORTS)), np.log10(Fg.loc[COHORTS]), marker="D", color="red", zorder=3, label="$F$")
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=60, ha="right", fontsize=8)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(xlabel="", ylabel=r"$\log_{10}\alpha_g$ (posterior median)")
ax.legend(fontsize=8, frameon=False)
sns.despine(); fig.tight_layout()
savefig(fig, "alpha_genes", "alpha_g_distribution_per_cohort")
plt.show()

# heatmap of r_g for the top over- and under-represented genes of every cohort (called genes, by r_g)
N_TOP = 6
top = []
for c in COHORTS:
    d = alpha_df.loc[alpha_df["cohort"] == c]
    top += d.loc[d["call"] == "over"].nlargest(N_TOP, "r_rel")["gene_id"].tolist() + d.loc[d["call"] == "under"].nsmallest(N_TOP, "r_rel")["gene_id"].tolist()
top = list(dict.fromkeys(top))
mat = alpha_df.pivot(index="gene_id", columns="cohort", values="r_rel").reindex(index=top, columns=COHORTS)
calls = alpha_df.pivot(index="gene_id", columns="cohort", values="call").reindex(index=top, columns=COHORTS)
save_tsv(mat.rename(index=gene_names).rename_axis("gene_name").reset_index(), "alpha_genes", "r_rel.top_genes_per_cohort.tsv")
if len(mat):
    lim = np.nanmax(np.abs(mat.values))
    col_colors = pd.DataFrame({"fraction": [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS],
                               "batch": [BATCH_COLORS[cohort_info.loc[c, "batch"]] for c in COHORTS]}, index=COHORTS)
    g = sns.clustermap(mat.fillna(0).rename(index=gene_names), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, col_cluster=False, row_cluster=len(mat) > 2,
                       col_colors=col_colors, annot=np.where(calls.isin(["over", "under"]), "*", ""), fmt="", annot_kws={"fontsize": 7},
                       figsize=(0.45 * len(COHORTS) + 4, 0.16 * len(mat) + 3), yticklabels=True, dendrogram_ratio=(0.1, 0.05),
                       cbar_kws={"label": r"$r_g = \log_2\alpha_g$ - cohort median"}, cbar_pos=(-0.08, 0.6, 0.02, 0.2))
    g.ax_heatmap.set(xlabel="", ylabel="")
    g.ax_heatmap.tick_params(axis="y", labelsize=6)
    color_ticklabels(g.ax_heatmap.get_xticklabels(), [cohort_info.loc[t.get_text(), "fraction"] for t in g.ax_heatmap.get_xticklabels()])
    g.savefig(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap.png"), bbox_inches="tight", dpi=200)
    g.savefig(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap.pdf"), bbox_inches="tight")
    figure_caption(os.path.join(FIGS, "alpha_genes", "r_rel.top_genes_per_cohort.clustermap"))
    plt.show()

In [ ]:
# gene biotypes: median r_g of the biotype's genes per cohort (all genes with alpha), and the number of genes
bt = alpha_df.groupby(["gene_type", "cohort"])["r_rel"].median().unstack().reindex(columns=COHORTS)
n_bt = alpha_df.loc[alpha_df["cohort"] == COHORTS[0], "gene_type"].value_counts()
bt = bt.loc[n_bt.index[n_bt >= 20]]
bt.index = [f"{b} (n={n_bt[b]:,})" for b in bt.index]
save_tsv(bt.rename_axis("gene_type").reset_index(), "alpha_genes", "r_rel.median_per_biotype.tsv")
lim = np.nanmax(np.abs(bt.values))
sns.set(font_scale=0.8, style="white")
fig, ax = plt.subplots(figsize=(0.5 * len(COHORTS) + 3, 0.3 * len(bt) + 1.5))
sns.heatmap(bt, cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, annot=True, fmt=".1f", annot_kws={"fontsize": 6}, ax=ax,
            cbar_kws={"label": "median $r_g$"})
ax.set(xlabel="", ylabel="")
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
savefig(fig, "alpha_genes", "r_rel.median_per_biotype")
plt.show()

# selected genes: compartment markers, nuclear RNAs, mitochondrial and housekeeping mRNAs
GOI = ["MALAT1", "NEAT1", "XIST", "SNHG1", "MT-CO1", "MT-ND1", "CALR", "HSPA5", "SSR4", "SEC61B", "CANX", "ATP5MC1", "COX7C", "TOMM20",
       "ACTB", "GAPDH", "RPLP0", "EEF1A1"]
genes_with_alpha = set(alpha_df["gene_id"])
name_to_id = {n: g for g, n in gene_names.items() if g in genes_with_alpha}
goi = [n for n in GOI if n in name_to_id]
ncols = 6
nrows = int(np.ceil(len(goi) / ncols))
sns.set(font_scale=0.8, style="ticks")
fig, axes = plt.subplots(nrows, ncols, figsize=(2.9 * ncols, 2.6 * nrows), sharey=True, squeeze=False)
for ax, n in zip(axes.flat, goi):
    d = alpha_df.loc[alpha_df["gene_id"] == name_to_id[n]].set_index("cohort").reindex(COHORTS)
    for i, (c, r) in enumerate(d.iterrows()):
        if pd.isna(r["alpha"]):
            continue
        ax.errorbar(i, r["alpha"], yerr=[[r["alpha"] - r["alpha_CI_lower"]], [r["alpha_CI_upper"] - r["alpha"]]], fmt=BATCH_MARKERS[cohort_info.loc[c, "batch"]],
                    color=FRACTION_COLORS[cohort_info.loc[c, "fraction"]], ms=5, ecolor="grey", elinewidth=0.8, capsize=1.5)
        ax.plot(i, Fg[c], marker="_", color="red", ms=8)
    ax.set_yscale("log")
    ax.set_xticks(range(len(COHORTS)), [cohort_info.loc[c, "fraction"] for c in COHORTS], rotation=90, fontsize=6)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
    ax.set_title(n, fontsize=9)
for ax in axes.flat[len(goi):]:
    ax.axis("off")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$\alpha_g$ (red: $F$)")
fig.legend(handles=notation_handles([], BATCH_ORDER[1:]), loc="upper right", fontsize=7, frameon=False)
fig.tight_layout()
savefig(fig, "alpha_genes", "selected_genes.alpha_per_cohort")
plt.show()

### Interpretation of $\alpha_g$

- **The model removes most of the strongest localization signal.** Genes far in the tails of $D_g$ are given a high outlier probability: 40% of the genes in the nuclear fraction (outlier probability > 0.5; $\pi$ = 0.5), 25% in NES, 15% in KDEL, 1% in ERM ($\pi$ = 0.03). An outlier's error variance is $V_g + \tau^2$ with $\tau$ = 2–4, so its posterior falls back to the fitted Beta and $r_g \approx 0$. Examples:
  - In APEX MITO, the mitochondrially encoded genes lie +5 log2 above the median gene ($D_g$ median 3.97 vs −1.17), but get $r_g$ = 0.007.
  - In the nuclear fraction, snoRNAs (+4.1) get $r_g$ = 0.011.
  - MALAT1 gets $r_g$ = −0.02 in NLS and −0.04 in Nuc.
  - The biotype heatmap is therefore almost blank, and the $r_g$ of the remaining genes are compressed (slope of $r_g$ on the centred $D_g$ 0.16–0.59; s.d. of $r_g$ 0.13 in KDEL, 0.88 in ERM).
- **Consequently there are no calls in NES, KDEL, Nuc and Cyto,** and calls mostly in the cohorts with a low $\pi$ or a wide Beta: ERM (1,775 over / 2,319 under; $\pi$ = 0.03, a = 2.2), NIK (1,025 / 1,011), OMM (508 / 2,092), LMA (91 / 2,743), NLS (9 / 2,001). Under-represented genes outnumber over-represented ones because the right tail of the log-Beta is lighter than the left.
- **What survives is consistent with the known localization:**
  - ERM and OMM are enriched for mRNAs of ER-membrane and secretory proteins: ERM TMEM30A, ESYT1, UGGT1, MAGT1, DHCR24, SCD, CALU; OMM P4HB, KDELR1, SLC39A14. CALR and HSPA5 are at +0.6 to +0.7 in ERM and depleted in NLS, NIK and LMA (−0.5 to −0.95).
  - OMM also gains nuclear-encoded mitochondrial mRNAs translated at the mitochondrial surface (ABCB10, TARS2).
  - Mitochondrially encoded genes are under-represented outside the mitochondria (ERM −0.7, OMM −0.6, NLS and LMA −0.4).
  - Abundant cytosolic mRNAs (GAPDH, ACTB, ribosomal proteins) are under-represented in all compartments except Cyto. This is the compression of the labelled libraries.
- **For APEX-seq, the $r_g$ of this model are usable for ranking within the bulk of genes, not for the most strongly localised RNAs.** A fit with a low bound on $\pi$ is tested against spike-in truth in `LoRNA_U2OS.fracSanity.ipynb` (fits "low pi").

## Functional groups over- and under-represented in each fraction (ISMARA)

ISMARA-style activities fitted on the **per-library** $\log_2\alpha_g$ (posterior means of fracSanity fits of each fraction library against the average of the controls; replicates are not paired with controls). Columns are centred (`center="columns"`): the activity of a set in a library is the shift of the $\log_2\alpha$ of its genes beyond the average gene of that library, independent of $F$. Activities are averaged over the libraries of a cohort (`reml`), and each cohort's average is tested against 0 (`activity_contrast`): **positive = the set's mRNAs are over-represented in the fraction** relative to the average mRNA, negative = under-represented.

As in `pbody_RNA_Spang.fracSanity.pcTPM.ipynb`, **one fit per database** (literature, GO:BP, GO:CC, GO:MF, Reactome, KEGG, WikiPathways; 5–500 genes per set, identical sets merged), each with two kinds of covariates: the gene biotypes (vs protein-coding genes) and the mitochondrially encoded genes (`BIOTYPE:`), and the deciles of expression in Total (`EXPR:decile01` = lowest, decile 5 as reference). As for the ONT fractions, the $\log_2\alpha_g$ of a fraction can depend on how abundant the mRNA is in the cell (`APEX_seq_vs_Control.ipynb`), and without the expression covariates, sets of abundant mRNAs (ribosomal proteins, translation, respiration) would appear under-represented for that reason alone. With them, a set's activity is its over- or under-representation **beyond what its genes' abundance and biotype predict**. The covariate rows are reported from the literature fit. `padj` is BH over the sets of one fit and cohort. Gene IDs: Ensembl gene IDs without version (GO symbols, KEGG and WikiPathways NCBI IDs translated with the NCBI `gene_info` and the annotation). Literature sets: cytoplasmic ribosomal-protein genes (`LIT:RP`) and replication-dependent histone genes (`LIT:histone_RD`, non-polyadenylated mRNAs).

In [ ]:
# per-library log2 alpha (posterior means), genes x libraries
lib_fits = {}
for s in meta.loc[meta["fraction"] != "Total", "sample"]:
    lib_fits[s] = load_pickle(os.path.join(SU_OUT, "fracSanity", "genes", "libraries", safe(s) + ".pkl"))
la = pd.DataFrame({s: pd.Series(f["log2_alpha_mean"], index=f["gene_ids"]) for s, f in lib_fits.items()})
lib_meta = meta.loc[meta["sample"].isin(la.columns), ["sample", "cohort"]].reset_index(drop=True)
F_lib = pd.DataFrame([{"sample": s, "cohort": f["cohort"], "F": f["F"], "F_CI_lower": f["F_CI_lower"], "F_CI_upper": f["F_CI_upper"],
                       "pi_outlier": f["pi_outlier"], "a": f["a"], "b": f["b"]} for s, f in lib_fits.items()])
save_tsv(F_lib, "gene_sets", "fracSanity_global_F.per_library.tsv")
la_out = la.copy()
la_out.insert(0, "gene_name", gene_names.reindex(la_out.index).to_numpy())
save_tsv(la_out.rename_axis("gene_id").reset_index(), "gene_sets", "fracSanity_log2_alpha.genes.per_library.tsv.gz")

# Ensembl IDs without version (PAR_Y copies removed)
values = la.loc[~la.index.str.endswith("_PAR_Y")]
values.index = values.index.str.replace(r"\.\d+$", "", regex=True)
assert values.index.is_unique
universe = set(values.index)
ens_of = pd.Series(genes_df.index[~genes_df.index.str.endswith("_PAR_Y")].str.replace(r"\.\d+$", "", regex=True),
                   index=genes_df.index[~genes_df.index.str.endswith("_PAR_Y")])
nm = gene_names.reindex(ens_of.index)
sym2ens = pd.Series(ens_of.to_numpy(), index=nm.to_numpy()).groupby(level=0).agg(list).to_dict()
gene_info = read_ncbi_gene_info(next(latest_release(GENESETS_DIR, "kegg").glob("*.gene_info*")))
ncbi2ens = {}
for gid, sym, xref in zip(gene_info["GeneID"].astype(str), gene_info["Symbol"], gene_info["dbXrefs"].fillna("")):
    e = [x.split(":", 1)[1] for x in xref.split("|") if x.startswith("Ensembl:")]
    ncbi2ens[gid] = e if e else sym2ens.get(sym, [])
ncbi2ens = {k: v for k, v in ncbi2ens.items() if v}

def prepare(coll):
    return coll.restrict_to(universe).filter_size(5, 500).drop_duplicates()

ens_name = pd.Series(nm.to_numpy(), index=ens_of.to_numpy())
lit = {"LIT:RP": {g for g, n in ens_name.items() if re.match(r"^RP[LS]\d+[A-Z]?\d*$", str(n)) and g in universe},
       "LIT:histone_RD": {g for g, n in ens_name.items() if re.match(r"^H(1-[0-6]|2AC\d+|2BC\d+|3C\d+|4C\d+)$", str(n)) and g in universe}}
db_colls = {"literature": GeneSetCollection(lit, {"LIT:RP": "cytoplasmic ribosomal-protein genes",
                                                  "LIT:histone_RD": "replication-dependent histone genes"}, "literature")}
for aspect in ("BP", "CC", "MF"):
    db_colls[f"GO:{aspect}"] = prepare(load_go(GENESETS_DIR, aspect=aspect, id_type="symbol").map_ids(sym2ens))
db_colls["Reactome"] = prepare(load_reactome(GENESETS_DIR))
db_colls["KEGG"] = prepare(load_kegg(GENESETS_DIR, id_map=ncbi2ens))
db_colls["WikiPathways"] = prepare(load_wikipathways(GENESETS_DIR, id_map=ncbi2ens))

bt_of = biotype.loc[ens_of.index]
bt_of.index = ens_of.to_numpy()
biotype_sets = {f"BIOTYPE:{b}": set(g.index) & universe for b, g in bt_of.groupby(bt_of) if b != "protein_coding"}
biotype_sets = {k: v for k, v in biotype_sets.items() if len(v) >= 5}
biotype_coll = GeneSetCollection(biotype_sets, {k: k.split(":")[1].replace("_", " ") + " (biotype)" for k in biotype_sets}, "biotype")
# expression covariates: deciles of the log2 TPM in Total (decile 5 is the reference and has no set)
tot_l2 = tot_av.means["Total"].loc[lambda x: ~x.index.str.endswith("_PAR_Y")]
tot_l2.index = tot_l2.index.str.replace(r"\.\d+$", "", regex=True)
decile = pd.qcut(tot_l2.reindex(sorted(universe)).dropna(), 10, labels=False)
expr_sets = {f"EXPR:decile{d + 1:02d}": set(decile.index[decile == d]) for d in range(10) if d != 4}
expr_coll = GeneSetCollection(expr_sets, {k: f"expression in Total, decile {k[-2:]}" for k in expr_sets}, "expression")
covariate_coll = biotype_coll + expr_coll
print(f"{values.shape[1]} libraries, {len(universe):,} genes; sets per database:", {k: len(v) for k, v in db_colls.items()},
      "| covariates:", len(covariate_coll))

In [ ]:
GENESET_DATABASES = list(db_colls)
act_rows, info_all, zm_all, lib_acts = [], [], [], []
for db, c_db in db_colls.items():
    coll_db = c_db + covariate_coll
    a = su.fit_set_activities(values, coll_db, center="columns", ridge="cv", cv_repeats=5)
    a_avg = a.average_conditions(lib_meta, sample_col="sample", cond_col="cohort", method="reml")
    info = set_source_table(coll_db).assign(fit=db)
    info["size"] = info["set_id"].map(coll_db.sizes)
    info["name"] = info["set_id"].map(coll_db.names)
    info_all.append(info)
    zm_all.append(info.assign(z_m=info["set_id"].map(a.z_values)))
    lib_acts.append(a.activities.assign(fit=db))
    for c in COHORTS:
        d = su.activity_contrast(a_avg, {c: 1.0})
        act_rows.append(d[["delta_activity", "SE", "z_score", "p_value", "padj", "lfsr"]].assign(cohort=c, fit=db).rename_axis("set_id").reset_index())
    print(f"{db}: {len(c_db)} sets (+ {len(covariate_coll)} covariates); lambda = {a.lambda_:.4g}; "
          f"median explained variance per library {a.explained_variance.median():.3f}")

COVARIATE = ("BIOTYPE:", "EXPR:")
keep = lambda df: df.loc[~df["set_id"].str.startswith(COVARIATE) | (df["fit"] == "literature")].reset_index(drop=True)
info_all = keep(pd.concat(info_all))
act_df = keep(pd.concat(act_rows)).rename(columns={"delta_activity": "activity"}).merge(info_all[["set_id", "fit", "database", "name", "size"]],
                                                                                         on=["set_id", "fit"], how="left")
act_df["database"] = np.select([act_df["set_id"].str.startswith("BIOTYPE:"), act_df["set_id"].str.startswith("EXPR:")], ["biotype", "expression"],
                               act_df["fit"])
zm_df = keep(pd.concat(zm_all)).sort_values("z_m", ascending=False).reset_index(drop=True)
save_tsv(act_df, "gene_sets", "gene_set_activities.per_cohort.tsv.gz")
save_tsv(zm_df, "gene_sets", "gene_set_activities.z_m.tsv")
lib_act_df = keep(pd.concat(lib_acts).rename_axis("set_id").reset_index())
save_tsv(lib_act_df, "gene_sets", "gene_set_activities.per_library.tsv.gz")
print(act_df.loc[act_df["padj"] < 0.05].groupby(["database", "cohort"]).size().unstack(fill_value=0).reindex(columns=COHORTS).to_string())

In [ ]:
set_short = act_df.drop_duplicates("set_id").set_index("set_id")["name"].fillna("").str.slice(0, 65)
col_colors = pd.DataFrame({"fraction": [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS],
                           "batch": [BATCH_COLORS[cohort_info.loc[c, "batch"]] for c in COHORTS]}, index=COHORTS)

def activity_clustermap(mat, padj, title, name, row_cluster=True, row_labels=None):
    # activities with * for padj < 0.05; columns: cohorts in batch order, colour bars for fraction and batch
    mat = mat.fillna(0)
    stars = np.where(padj.reindex(index=mat.index, columns=mat.columns) < 0.05, "*", "")
    h = 0.2 * len(mat) + 3
    lim = np.nanmax(np.abs(mat.values))
    sns.set(font_scale=0.8, style="white")
    g = sns.clustermap(mat.rename(index=row_labels or {}), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, row_cluster=row_cluster and len(mat) > 2,
                       col_cluster=False, method="average", col_colors=col_colors.reindex(mat.columns), colors_ratio=(0.03, 0.3 / h),
                       annot=stars, fmt="", annot_kws={"fontsize": 8}, figsize=(0.45 * mat.shape[1] + 7, h), yticklabels=True,
                       cbar_kws={"label": "activity (over- / under-representation)"}, dendrogram_ratio=(0.08, 0.6 / h),
                       cbar_pos=(-0.08, 0.6, 0.02, min(0.25, 2 / h)))
    g.ax_heatmap.set(xlabel="", ylabel="")
    color_ticklabels(g.ax_heatmap.get_xticklabels(), [cohort_info.loc[t.get_text(), "fraction"] for t in g.ax_heatmap.get_xticklabels()])
    g.ax_col_dendrogram.set_title(title, fontsize=9)
    out = os.path.join(FIGS, "gene_sets")
    os.makedirs(out, exist_ok=True)
    g.savefig(os.path.join(out, f"{name}.pdf"), bbox_inches="tight")
    g.savefig(os.path.join(out, f"{name}.png"), bbox_inches="tight", dpi=200)
    figure_caption(os.path.join(out, name))
    plt.show()

# per database: sets with padj < 0.05 in at least one cohort (at most MAX_ROWS, smallest padj), rows clustered
MAX_ROWS = 50
for db in GENESET_DATABASES + ["biotype", "expression"]:
    d = act_df.loc[act_df["database"] == db]
    sig = d.loc[d["padj"] < 0.05, "set_id"].unique()
    if len(sig) == 0:
        print(f"{db}: no set with padj < 0.05")
        continue
    mat = d.loc[d["set_id"].isin(sig)].pivot(index="set_id", columns="cohort", values="activity")[COHORTS]
    padj = d.loc[d["set_id"].isin(sig)].pivot(index="set_id", columns="cohort", values="padj")[COHORTS]
    save_tsv(mat.rename_axis("set_id").reset_index().assign(name=lambda x: x["set_id"].map(set_short)),
             "gene_sets", f"gene_set_activities.sig_in_any_cohort.{db.replace(':', '_')}.tsv")
    shown = padj.min(axis=1).sort_values().index[:MAX_ROWS]
    activity_clustermap(mat.loc[shown], padj.loc[shown], f"{db}: {len(shown)} of {len(sig)} sets with padj < 0.05 in at least one cohort",
                        f"activities.sig_in_any_cohort.{db.replace(':', '_')}", row_labels=set_short.to_dict())

### Sets ordered by their overall significance $z_m$

Per database, the 12 sets with the highest $z_m$ over all libraries (rows sorted by $z_m$); colour: cohort-averaged activity, \* padj < 0.05.

In [ ]:
N_TOP_ZM = 12
for db in [d for d in GENESET_DATABASES if d != "literature"]:
    top_zm = zm_df.loc[zm_df["fit"] == db].loc[lambda x: ~x["set_id"].str.startswith(COVARIATE)].head(N_TOP_ZM)
    if top_zm.empty:
        continue
    d = act_df.loc[act_df["set_id"].isin(top_zm["set_id"]) & (act_df["fit"] == db)]
    mat = d.pivot(index="set_id", columns="cohort", values="activity").reindex(index=top_zm["set_id"], columns=COHORTS)
    padj = d.pivot(index="set_id", columns="cohort", values="padj").reindex(index=top_zm["set_id"], columns=COHORTS)
    labels = {s: f"{set_short.get(s, s)}  z_m={z:.1f}" for s, z in zip(top_zm["set_id"], top_zm["z_m"])}
    activity_clustermap(mat, padj, f"{db}: top {len(mat)} sets by $z_m$", f"activities.top_z_m.{db.replace(':', '_')}", row_cluster=False, row_labels=labels)

### Interpretation of the gene-set analysis

- **OMM, NES, ERM and NLS have the most sets** at padj < 0.05 (e.g. GO:CC 29 / 35 / 15 / 11, KEGG 61 / 52 / 31 / 33). NIK, NucPore, KDEL and MITO have almost none (0–10 per database). In KDEL and MITO this is because their $\log_2\alpha$ are shrunk most (previous section), and in NIK and NucPore because of their low library complexity.
- **ER-translated mRNAs.** "Endoplasmic reticulum lumen" and "ER protein-containing complex", the two sets with the highest $z_m$ over all cohorts, are over-represented in ERM and OMM, slightly also in the nuclear fraction, and under-represented in NES, LMA, NLS and Cyto. ERM labels the cytosolic face of the ER, where these mRNAs are translated, and OMM labels ER–mitochondria contacts, as in Fazal et al. (2019). In the nuclear fraction they most likely come from the rough ER attached to the nuclear envelope, which sediments with nuclei.
- **Under-represented in ERM, OMM and NES:** mitochondrial inner membrane, respiratory chain, ribosomal subunit and chaperone sets, i.e. the abundant cytosolically translated mRNAs. The spliceosomal complex is over-represented in NLS, NIK and LMA and in the nuclear fraction.
- **Covariates.** Biotype and expression covariates are significant in most cohorts (biotype 1–11, expression 0–9 per cohort), as in the ONT fractions.

## Tandem poly(A) sites of terminal exons

PAQR (`Snakefile-quantification-faster`, `Snakefile-PAQR-quantify`; PolyASite 3.0 clusters on the terminal exons of GENCODE v42, `atlas.clusters.3.0.GRCh38.GENCODE_42.terminal_exons.bed`): the expression of each tandem PAS from the drop of the read coverage at the site (`tandem_pas_expression-mean`), per library. Terminal exons with a missing value (−1, no reliable coverage profile) in any library are left out; PAS with ≥ 5 (rounded coverage, summed over the libraries) enter the analyses. **3' UTR length** of a PAS: distance from the stop codon of the Ensembl canonical transcript (+1) when the stop codon lies in the terminal exon, otherwise from the 5' end of the terminal exon (as for the ONT cleavage sites; only differences within a gene matter). Short-read coverage cannot resolve sites closer than the fragment length, and PAQR merges PolyASite clusters within the cluster distance; the PAS here are therefore coarser than the read 3' ends of the ONT data.

In [ ]:
pas_tab = pd.read_csv(os.path.join(SU_OUT, "inputs", "pas_table.tsv.gz"), sep="\t", index_col=0)
pas_counts = pd.read_csv(os.path.join(SU_OUT, "inputs", "pas_counts.tsv.gz"), sep="\t", index_col=0)   # PAS with >= 5
used = pas_tab.loc[pas_tab["passes_min_reads"]]
n_pas = used.groupby("exon").size()
print(f"{len(pas_tab):,} tandem PAS in {pas_tab['exon'].nunique():,} terminal exons of {pas_tab['gene_id'].nunique():,} genes; "
      f"{len(used):,} PAS with >= 5 in {used['exon'].nunique():,} terminal exons ({(n_pas >= 2).sum():,} with >= 2 PAS); "
      f"3' UTR length from the stop codon for {(used['utr_origin'] == 'stop_codon').mean():.1%}")
save_tsv(used.reset_index(), "tandem_pas", "pas_table.min5.tsv.gz")

lib_order = [s for c in ["Total"] + COHORTS for s in meta.loc[meta["cohort"] == c, "sample"]]
sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].hist(np.log10(used["utr_length"]), bins=50, color="tab:blue")
axes[0].set(xlabel="log10 3' UTR length of the PAS (nt)", ylabel="PAS")
axes[1].bar(*np.unique(np.clip(n_pas, 1, 8), return_counts=True), color="tab:blue")
axes[1].set(xlabel="PAS per terminal exon (8 = 8 or more)", ylabel="terminal exons")
prox = used.loc[used["polyAsite_exon_idx"] == 1].index
share_prox = (pas_counts.loc[pas_counts.index.intersection(prox), lib_order].sum() / pas_counts[lib_order].sum())
axes[2].bar(np.arange(len(lib_order)), share_prox, color=[FRACTION_COLORS[f] for f in meta.set_index("sample").loc[lib_order, "fraction"]])
axes[2].set_xticks(np.arange(len(lib_order)), lib_order, rotation=90, fontsize=5.5)
color_ticklabels(axes[2].get_xticklabels(), meta.set_index("sample").loc[lib_order, "fraction"])
axes[2].set(ylabel="share of PAS signal at the\nmost proximal PAS of the exon")
sns.despine(); fig.tight_layout()
savefig(fig, "tandem_pas", "pas_utr_length.n_sites.proximal_share")
plt.show()

## Relative usage of tandem PAS

Joint Sanity run (EAP) on the PAS of controls and fractions. For every PAS of a terminal exon with at least two PAS: log2 odds of its usage against the other PAS of the exon, per library, averaged per cohort (`reml`); each cohort compared with the controls (`differential_relative_usage`), $\Delta\Delta$ = change of the log2 odds, BH-adjusted p-values.

In [ ]:
usage = load_pickle(os.path.join(SU_OUT, f"usage_{PAS_LEVEL}", "pas_usage.per_sample.pkl"))
u_means, u_errors, u_tau = usage["averages"]
u_tab = used[["gene_id", "gene_name", "exon", "polyAsite_exon_idx", "nr_polyAsites_on_exon", "utr_length", "pas_pos", "strand"]].join(
    (2 ** u_means / (1 + 2 ** u_means)).add_prefix("usage;"), how="inner").join(u_means.add_prefix("log2_odds;")).join(u_errors.add_prefix("SE;"))
save_tsv(u_tab.rename_axis("pas").reset_index(), "pas_usage", "pas_relative_usage.per_cohort.tsv.gz")
print(f"{len(u_tab):,} PAS in {u_tab['exon'].nunique():,} terminal exons with >= 2 PAS")

du = {}
for c in COHORTS:
    d = pd.read_csv(os.path.join(SU_OUT, f"usage_{PAS_LEVEL}", f"pas_usage.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
    du[c] = d.join(used[["gene_id", "polyAsite_exon_idx", "nr_polyAsites_on_exon", "utr_length"]])
du_long = pd.concat([d.rename(columns=lambda x: x.replace(f"_{c}", "_fraction")).assign(cohort=c) for c, d in du.items()]).rename_axis("pas").reset_index()
du_long.insert(2, "gene_name", gene_names.reindex(du_long["gene_id"]).to_numpy())
save_tsv(du_long, "pas_usage", "pas_relative_usage.fraction_vs_Total.tsv.gz")
n_du = pd.DataFrame([{"cohort": c, "n_tests": len(d), "higher": int(((d["padj"] < 0.05) & (d["dd_log2FC"] > 0)).sum()),
                      "lower": int(((d["padj"] < 0.05) & (d["dd_log2FC"] < 0)).sum())} for c, d in du.items()]).set_index("cohort")
save_tsv(n_du.reset_index(), "pas_usage", "pas_relative_usage.fraction_vs_Total.n_significant.tsv")
display(n_du)

## 3' UTR length (tandem APA) in the fractions relative to the controls

For every gene with at least two PAS in its terminal exon (the terminal exon with the most signal if a gene has several): the usage-weighted mean 3' UTR length $L_g$ of each library (`utr_length_index`), averaged per cohort; $D_g = \langle\log_2 L_g\rangle_{\text{fraction}} - \langle\log_2 L_g\rangle_{\text{controls}}$ (`differential_utr_length`), shortening / lengthening at padj < 0.05; two-site view: distal PAS against the most proximal one (`differential_proximal_distal`, `splits="first"`; $\Delta\Delta > 0$ = more distal usage in the fraction).

In [ ]:
apa, pdist = {}, {}
for c in COHORTS:
    apa[c] = pd.read_csv(os.path.join(SU_OUT, f"usage_{PAS_LEVEL}", f"utr_length.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
    pdist[c] = pd.read_csv(os.path.join(SU_OUT, f"usage_{PAS_LEVEL}", f"proximal_vs_distal.{safe(c)}_vs_Total.tsv.gz"), sep="\t", index_col=0)
ren = lambda d, c: d.rename(columns=lambda x: x.replace(f"_{c}", "_fraction"))
apa_long = pd.concat([ren(d, c).assign(cohort=c) for c, d in apa.items()]).rename_axis("gene_id").reset_index()
apa_long.insert(1, "gene_name", gene_names.reindex(apa_long["gene_id"]).to_numpy())
save_tsv(apa_long, "apa", "utr_length.fraction_vs_Total.tsv.gz")
save_tsv(pd.concat([ren(d, c).assign(cohort=c) for c, d in pdist.items()]).rename_axis("group").reset_index(), "apa",
         "proximal_vs_distal.fraction_vs_Total.tsv.gz")
uli = load_pickle(os.path.join(SU_OUT, f"usage_{PAS_LEVEL}", "utr_length_index.per_sample.pkl"))
li_means, li_errors, _ = uli["averages"]
li = li_means.add_prefix("log2_L;").join(li_errors.add_prefix("SE;"))
li.insert(0, "gene_name", gene_names.reindex(li.index).to_numpy())
save_tsv(li.rename_axis("gene_id").reset_index(), "apa", "utr_length_index.log2_mean_3UTR_length.per_cohort.tsv.gz")

summary = []
for c in COHORTS:
    d, p = apa[c], pdist[c]
    pp = p.join(d["log2FC_length"], on="gene")
    summary.append({"cohort": c, "genes": len(d), "shortening": int((d["call"] == "shortening").sum()), "lengthening": int((d["call"] == "lengthening").sum()),
                    "median_log2FC_length": d["log2FC_length"].median(),
                    "median_log2FC_length_padj<0.05": d.loc[d["padj"] < 0.05, "log2FC_length"].median(),
                    "spearman_length_vs_distal_proximal": spearmanr(pp["log2FC_length"], pp["dd_log2FC"], nan_policy="omit")[0]})
apa_summary = pd.DataFrame(summary).set_index("cohort")
save_tsv(apa_summary.reset_index(), "apa", "apa_summary.fraction_vs_Total.tsv")
display(apa_summary.round(3))

sns.set(font_scale=0.85, style="ticks")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 1.4]})
x = np.arange(len(COHORTS))
cols = [FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS]
ax = axes[0]
ax.bar(x, -apa_summary.loc[COHORTS, "shortening"], color=cols, edgecolor="black", lw=0.4, hatch="//")
ax.bar(x, apa_summary.loc[COHORTS, "lengthening"], color=cols, edgecolor="black", lw=0.4)
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="# genes: lengthening (up) /\nshortening (down) vs controls, padj < 0.05")
ax = axes[1]
sns.boxplot(data=apa_long, x="cohort", y="log2FC_length", order=COHORTS, hue="cohort", palette=dict(zip(COHORTS, cols)), showfliers=False, ax=ax,
            legend=False, width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.set(ylabel="$D_g$ = log2 mean 3' UTR length,\nfraction - controls (all tested genes)", xlabel="")
for ax in axes:
    ax.set_xticks(x, COHORTS, rotation=90, fontsize=7)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
sns.despine(); fig.tight_layout()
savefig(fig, "apa", "apa.fraction_vs_Total.summary")
plt.show()

calls = apa_long.pivot(index="gene_id", columns="cohort", values="call").reindex(columns=COHORTS)
n_called = (calls != "NS").sum(axis=1)
top_genes = n_called.sort_values(ascending=False).index[:40]
mat = apa_long.pivot(index="gene_id", columns="cohort", values="log2FC_length").reindex(index=top_genes, columns=COHORTS)
if len(mat) and n_called.max() > 0:
    lim = np.nanmax(np.abs(mat.values))
    sns.set(font_scale=0.8, style="white")
    fig, ax = plt.subplots(figsize=(0.45 * len(COHORTS) + 3, 0.18 * len(mat) + 1.5))
    sns.heatmap(mat.rename(index=gene_names), cmap="RdBu_r", center=0, vmin=-lim, vmax=lim, ax=ax, yticklabels=True,
                annot=np.where(calls.reindex(index=top_genes) != "NS", "*", ""), fmt="", annot_kws={"fontsize": 7},
                cbar_kws={"label": "$D_g$ (log2 mean 3' UTR length, fraction - controls)"})
    ax.set(xlabel="", ylabel="", title="40 genes called in most cohorts (* padj < 0.05)")
    ax.tick_params(axis="y", labelsize=6)
    color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
    savefig(fig, "apa", "apa.top_genes.D_heatmap")
    plt.show()

In [ ]:
# APA shift and nuclear RNA: MALAT1 read pairs per read pair on nuclear protein-coding genes, relative to the controls
nuc_pc = genes_df.index[(genes_df["gene_type"] == "protein_coding") & (genes_df["chrom"] != "chrM")]
malat1 = genes_df.index[genes_df["gene_name"] == "MALAT1"]
gc_m, gc_n = gene_counts.reindex(malat1, fill_value=0), gene_counts.reindex(nuc_pc, fill_value=0)
tot_s = meta.loc[meta["fraction"] == "Total", "sample"]
R_malat1 = (gc_m.sum() / gc_n.sum()) / (gc_m[tot_s].sum().sum() / gc_n[tot_s].sum().sum())
R_cohort = np.exp(np.log(R_malat1.reindex(meta["sample"]).clip(lower=1e-2)).groupby(meta.set_index("sample")["cohort"]).mean())
apa_summary["R_MALAT1"] = R_cohort.reindex(apa_summary.index)
fig, ax = plt.subplots(figsize=(4.6, 3.8))
for c, r in apa_summary.iterrows():
    ax.scatter(np.log2(r["R_MALAT1"]), r["median_log2FC_length"], color=FRACTION_COLORS[cohort_info.loc[c, "fraction"]],
               marker=BATCH_MARKERS[cohort_info.loc[c, "batch"]], s=50, edgecolor="black", lw=0.4)
    ax.annotate(cohort_info.loc[c, "fraction"], (np.log2(r["R_MALAT1"]), r["median_log2FC_length"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
rho, p = spearmanr(np.log2(apa_summary["R_MALAT1"]), apa_summary["median_log2FC_length"])
ax.axhline(0, color="grey", lw=0.8)
ax.set(xlabel="nuclear RNA, log2 R MALAT1", ylabel="median $D_g$ (3' UTR length,\nfraction - controls)",
       title=f"Spearman {rho:.2f} (p = {p:.2g}), n = {len(apa_summary)}")
sns.despine(); fig.tight_layout()
savefig(fig, "apa", "apa.median_D_vs_nuclear_RNA")
plt.show()

### Interpretation of the tandem PAS usage and APA

- **Coverage of the terminal exons.** PAQR quantified 11,219 tandem PAS in 3,716 terminal exons. 1,835 exons were left out because PAQR reported −1 (no reliable coverage profile) in at least one of the 40 libraries, mostly because of NIK T1, OMM C2 and MITO T2tech. That leaves 6,034 PAS in 1,881 exons, of which 1,814 have ≥ 2 PAS (5,725 tests). **The APA results describe half of the tandem-APA genes, biased towards well-covered ones.** Excluding the worst libraries, or allowing a missing value per library, would recover most of the rest.
- **Relative usage differs from the controls in every cohort,** most in the nuclear fraction (1,685 PAS), NES (1,535), NIK (1,443) and NLS (1,267), least in MITO (234).
- **3' UTRs are longer than in the controls in most cohorts:**
  - lengthening dominates in NLS (594 genes lengthening / 9 shortening), NES (546 / 21), Nuc (549 / 38), LMA (481 / 12), KDEL (300 / 4), Cyto (237 / 14) and OMM (211 / 24);
  - shortening dominates in NIK (101 / 457) and NucPore (50 / 215);
  - ERM is balanced (139 / 156).

  The two-site view agrees (Spearman of $D_g$ with the distal-vs-proximal change 0.36–0.70).
- **Both fractionation compartments are longer than the controls** (median $D_g$ Nuc +0.115, Cyto +0.062). The nuclear-vs-cytosolic difference (+0.05) is small against the difference of both from the controls. The controls (whole cells of the APEX constructs) and the fractionation libraries are separate preparations, so a large part of the shift is not localization.
- **The APA shift follows library complexity, not nuclear RNA.** Across the 9 APEX constructs, the median $D_g$ correlates with the mean log2 complexity of the libraries relative to the controls (Spearman 0.80, p = 0.01; 0.62, p = 0.04, with the two fractionation cohorts). It does not correlate with MALAT1 $R$ (0.18, p = 0.59, figure above): NIK and NucPore are nuclear constructs with shortening, NES a cytosolic one with lengthening. NIK, NucPore and ERM, the constructs without lengthening, have the lowest complexity (1.7–3.7% distinct fragments). PCR duplicates make the coverage profiles at the 3' end noisy, and PAQR quantifies the PAS from coverage drops. **Unlike in the ONT fractions, the APA differences of APEX-seq cannot be attributed to compartments without a correction for complexity.** One option is to quantify after removing duplicates.

## fracSanity on tandem PAS: $\alpha$ of every 3' end isoform

fracSanity per cohort on the PAS (Sanity of controls and fractions separately, as for genes): $\alpha_j$ of every PAS $j$. For two PAS of one terminal exon, $\log_2\alpha_j - \log_2\alpha_k$ does not depend on $F$. Below, for terminal exons with at least two PAS: $\Delta = \log_2\alpha_{\text{distal}} - \log_2\alpha_{\text{proximal}}$ (longest vs shortest 3' UTR among the PAS of the exon in the fit; posterior means), per cohort, compared with $D_g$ of the APA analysis.

In [ ]:
pas_alpha = []
for c, f in fits[PAS_LEVEL].items():
    pas_alpha.append(pd.DataFrame({"pas": f["gene_ids"], "alpha": f["alpha_g"], "alpha_CI_lower": f["alpha_g_CI_lower"],
                                   "alpha_CI_upper": f["alpha_g_CI_upper"], "log2_alpha_mean": f["log2_alpha_mean"],
                                   "log2_alpha_sd": np.sqrt(f["log2_alpha_var"])}).assign(cohort=c))
pas_alpha = pd.concat(pas_alpha, ignore_index=True).merge(used[["gene_id", "gene_name", "exon", "utr_length", "pas_pos"]], left_on="pas",
                                                          right_index=True, how="left")
save_tsv(pas_alpha, "alpha_pas", "fracSanity_alpha.pas.per_cohort.tsv.gz")

grp = pas_alpha.sort_values("utr_length").groupby(["cohort", "exon"])
prox, dist, n_sites = grp.first(), grp.last(), grp.size()
dd_pas = pd.DataFrame({"gene_id": dist["gene_id"], "n_sites": n_sites, "delta_log2_alpha_distal_proximal": dist["log2_alpha_mean"] - prox["log2_alpha_mean"],
                       "log2_length_ratio": np.log2(dist["utr_length"] / prox["utr_length"])}).loc[n_sites >= 2].reset_index()
dd_pas = dd_pas.merge(apa_long[["gene_id", "cohort", "log2FC_length"]], on=["gene_id", "cohort"], how="left")
save_tsv(dd_pas, "alpha_pas", "delta_log2_alpha.distal_vs_proximal.per_exon.tsv.gz")
s = dd_pas.groupby("cohort").apply(lambda d: pd.Series({"exons": len(d), "median_delta": d["delta_log2_alpha_distal_proximal"].median(),
                                                        "share_positive": (d["delta_log2_alpha_distal_proximal"] > 0).mean(),
                                                        "spearman_vs_D_length": spearmanr(d["delta_log2_alpha_distal_proximal"], d["log2FC_length"],
                                                                                          nan_policy="omit")[0]})).reindex(COHORTS)
display(s.round(3))
save_tsv(s.reset_index(), "alpha_pas", "delta_log2_alpha.distal_vs_proximal.summary.tsv")

sns.set(font_scale=0.85, style="ticks")
fig, ax = plt.subplots(figsize=(8, 4.0))
sns.boxplot(data=dd_pas, x="cohort", y="delta_log2_alpha_distal_proximal", order=COHORTS, hue="cohort",
            palette={c: FRACTION_COLORS[cohort_info.loc[c, "fraction"]] for c in COHORTS}, showfliers=False, ax=ax, legend=False, width=0.6)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(range(len(COHORTS)), COHORTS, rotation=90, fontsize=7)
color_ticklabels(ax.get_xticklabels(), [cohort_info.loc[c, "fraction"] for c in COHORTS])
ax.set(xlabel="", ylabel=r"$\log_2\alpha_{distal} - \log_2\alpha_{proximal}$")
sns.despine(); fig.tight_layout()
savefig(fig, "alpha_pas", "delta_log2_alpha.distal_vs_proximal")
plt.show()

### Interpretation of the PAS-level fracSanity

- **$F$ is not identified at the PAS level either,** and $\pi$ is at its bound in 6 of 11 fits, so the within-exon differences of $\log_2\alpha$ are shrunk like the gene-level $r_g$.
- **Within exons, the distal isoform is enriched in most cohorts:** median $\log_2\alpha_{distal} - \log_2\alpha_{proximal}$ +0.58 (NES), +0.42 (NLS), +0.39 (LMA), +0.37 (Nuc), +0.34 (OMM), +0.28 (Cyto), +0.25 (KDEL), +0.21 (MITO), +0.12 (ERM). It is −0.20 in NIK and −0.01 in NucPore. This is the same order as the APA analysis (Spearman with $D_g$ 0.33–0.65), and therefore subject to the same complexity confounder.

## Summary

- **Data.** 41 APEX-seq libraries (Fazal et al. 2019, HEK293T) through the bulk workflow: 9 APEX constructs (2 targets each, MITO with a technical replicate), 18 unlabelled controls, nuclear and cytosolic fractionation. 29,966 genes and 5,725 tandem PAS (1,814 terminal exons) in the analyses.
- **fracSanity on APEX-seq.** $F$ is not identified (as expected, it has no physical meaning for biotinylation). With the outlier fraction up to 0.5, the model discounts the most strongly localised RNAs as outliers, e.g. mitochondrial transcripts in MITO and snoRNAs and MALAT1 in the nucleus, and compresses the rest. Calls are possible only where $\pi$ stays small (ERM, NIK, OMM, LMA, NLS). The surviving signal agrees with known localization: secretory and ER-membrane mRNAs in ERM and OMM, mitochondrial transcripts depleted outside mitochondria, spliceosome mRNAs in the nuclear constructs.
- **APA.** Longer 3' UTRs than in the controls in most cohorts, including both fractionation compartments, shortening in NIK and NucPore. The shift follows library complexity (Spearman 0.80), so it reflects the libraries more than the compartments. Half of the terminal exons were lost to PAQR's −1 values.
- **For fracSanity in general:** the outlier component should not absorb strongly localised genes, and $F$ needs external information. Both are tested with spike-in truth in `LoRNA_U2OS.fracSanity.ipynb`.